In [99]:
from tqdm.autonotebook import tqdm
import pandas as pd
import numpy as np
from pathlib import Path
import faiss           

In [100]:
res = faiss.StandardGpuResources()  # use a single GPU

In [88]:
from pathlib import Path
# embeddings_folder = Path("/opt/home/cleong/data/semlex_asl_citizen_popsign_combined_mini")
embeddings_folder = Path("/opt/home/cleong/data/ASL_Citizen/embeddings")
semlex_model_embeddings = list(embeddings_folder.rglob("*sem-lex*.npy"))


In [101]:
len(semlex_model_embeddings)

83112

In [90]:
def load_pose_embedding(embedding_path):
    embeddings = np.load(embedding_path).squeeze()
    # print(f"loaded embeddings with shape {embeddings.shape}")  # (1, 768)
    #   print(f"{embeddings[0]}=") # big long bunch of numbers
    # print(f"loaded embeddings with shape {embeddings}")
    return embeddings

In [80]:
# def list_of_paths_to_np_array(paths):
#     return np.array([load_pose_embedding(p) for p in tqdm(paths, desc="loading paths")])

In [108]:
def paths_to_dataframe(paths):
    data = [
        {
            "video_id": p.name.split("-")[0],
            "file_path": p,
            "file_name": Path(p).name,
            "embedding": load_pose_embedding(p),
        
        }
        for p in tqdm(paths, desc="loading paths")
    ]
    return pd.DataFrame(data)

In [109]:
split_index = 20
reference_paths = semlex_model_embeddings[split_index:]
query_paths = semlex_model_embeddings[:split_index]

In [110]:
query_df = paths_to_dataframe(query_paths)
query_df

loading paths: 100%|██████████| 20/20 [00:00<00:00, 4380.93it/s]


,video_id,file_path,file_name,embedding
0,4512868342911205,/opt/home/cleong/data/ASL_Citizen/embeddings/4...,4512868342911205-RESPONSIBILITY-using-model-se...,"[-0.33322427, 0.16524771, 0.11535681, 0.368136..."
1,802432162454322,/opt/home/cleong/data/ASL_Citizen/embeddings/8...,802432162454322-VITAMINS-using-model-sem-lex.npy,"[-0.23754077, 0.3569193, -0.3196658, 0.1353952..."
2,29766213567797584,/opt/home/cleong/data/ASL_Citizen/embeddings/2...,29766213567797584-ENVELOPE-using-model-sem-lex...,"[0.18618162, 0.1113351, -0.072578825, -0.10251..."
3,1989909351732737,/opt/home/cleong/data/ASL_Citizen/embeddings/1...,1989909351732737-FUNCTION-using-model-sem-lex.npy,"[-0.024617223, -0.20148936, -0.0022485945, 0.2..."
4,7075908041208987,/opt/home/cleong/data/ASL_Citizen/embeddings/7...,7075908041208987-STORY_3-using-model-sem-lex.npy,"[-0.3644051, 0.06412346, 0.020423084, 0.035708..."
5,08540962663657381,/opt/home/cleong/data/ASL_Citizen/embeddings/0...,08540962663657381-TIE-using-model-sem-lex.npy,"[0.07949132, -0.15333198, 0.06991726, 0.131780..."
6,029833014387566248,/opt/home/cleong/data/ASL_Citizen/embeddings/0...,029833014387566248-SAIL_2-using-model-sem-lex.npy,"[-0.20589952, -0.4929245, -0.405769, 0.4774660..."
7,8555548615892776,/opt/home/cleong/data/ASL_Citizen/embeddings/8...,8555548615892776-HOE-using-model-sem-lex.npy,"[-0.042334426, -1.0742257, -0.4117589, 0.04502..."
8,2753221560775323,/opt/home/cleong/data/ASL_Citizen/embeddings/2...,2753221560775323-RACE-using-model-sem-lex.npy,"[0.15693352, -0.44518018, -0.010830048, 0.2055..."
9,484005199725384,/opt/home/cleong/data/ASL_Citizen/embeddings/4...,484005199725384-LEAF_2-using-model-sem-lex.npy,"[0.48093784, -0.15828885, -0.20807397, 0.33676..."


In [113]:
def get_embeddings_array_from_df(df):
    embeddings_array = np.stack(df["embedding"].values)
    return embeddings_array

In [114]:
get_embeddings_array_from_df(query_df).shape

(20, 768)

In [115]:
ref_df = paths_to_dataframe(reference_paths)

loading paths: 100%|██████████| 83092/83092 [14:00<00:00, 98.86it/s]  


In [120]:
recombined_df = pd.concat([query_df, ref_df], ignore_index=True)

https://www.pinecone.io/learn/series/faiss/faiss-tutorial/ shows how to do this with a pandas dataframe

In [121]:
signclip_embedding_d = 768
d = signclip_embedding_d                           # dimension

nb = len(reference_paths)                      # database size
nq = len(query_paths)                      # nb of queries
np.random.seed(1234)             # make reproducible
# xb = get_embeddings_array_from_df(ref_df)
xb = get_embeddings_array_from_df(recombined_df)
xq = get_embeddings_array_from_df(query_df)

xb.shape, xq.shape

((83112, 768), (20, 768))

In [122]:

index = faiss.IndexFlatIP(d)
index = faiss.index_cpu_to_gpu(res, 0, index)
print(index.is_trained)
faiss.normalize_L2(xb) # normalize
index.add(xb)                  # add vectors to the index
print(index.ntotal)

True
83112


In [126]:
# data['sentence_A'].iloc[[4586, 10252, 12465, 190]]
recombined_df.iloc[[0, 3]]

,video_id,file_path,file_name,embedding
0,4512868342911205,/opt/home/cleong/data/ASL_Citizen/embeddings/4...,4512868342911205-RESPONSIBILITY-using-model-se...,"[-0.33322427, 0.16524771, 0.11535681, 0.368136..."
3,1989909351732737,/opt/home/cleong/data/ASL_Citizen/embeddings/1...,1989909351732737-FUNCTION-using-model-sem-lex.npy,"[-0.024617223, -0.20148936, -0.0022485945, 0.2..."


In [139]:
def search_and_output_indices(query_df, k=4):
    xq = get_embeddings_array_from_df(query_df)
    
    # xq = list_of_paths_to_np_array(query_paths)
    faiss.normalize_L2(xq) # normalize query vectors before search
    D, I = index.search(xq, k)     # Distances and Indexes of neighbors
    # print(I[:5])                   # neighbors of the 5 first queries
    print()
    for q_index, neighbors in enumerate(I):
        print(q_index)
        query_file = query_df['file_name'].iloc[q_index]

        # print(f"Query Path: {query_df['file_name'][q_index].name}")
        print(f"Query File: {query_file}")
        # print(query_file)
        for k_index, neighbor in enumerate(neighbors):
            
            
            distance_to_neighbor = D[q_index][k_index]
            db_index_of_neighbor = I[q_index][k_index]
            # print(f"{distance_to_neighbor} similarity")
            # print("Result File")
            result_file = recombined_df['file_name'].iloc[db_index_of_neighbor]
            print("*", distance_to_neighbor,result_file)
    # print(I[-5:])                  # neighbors of the 5 last queries
# search_and_output_original_paths(reference_paths[:5])
    return I


search_and_output_indices(recombined_df.head())


0
Query File: 4512868342911205-RESPONSIBILITY-using-model-sem-lex.npy
* 0.99999994 4512868342911205-RESPONSIBILITY-using-model-sem-lex.npy
* 0.94378984 05789382245740726-RESPONSIBILITY-using-model-sem-lex.npy
* 0.9384923 37172909077776795-RESPONSIBILITY-using-model-sem-lex.npy
* 0.9375743 10866847898271859-RESPONSIBILITY-using-model-sem-lex.npy
1
Query File: 802432162454322-VITAMINS-using-model-sem-lex.npy
* 1.0000001 802432162454322-VITAMINS-using-model-sem-lex.npy
* 0.97884727 36886063518419365-VANILLA-using-model-sem-lex.npy
* 0.95855355 1201537952691305-GOTCHA-using-model-sem-lex.npy
* 0.9569726 4150403650861141-VANILLA-using-model-sem-lex.npy
2
Query File: 29766213567797584-ENVELOPE-using-model-sem-lex.npy
* 0.99999994 29766213567797584-ENVELOPE-using-model-sem-lex.npy
* 0.8125504 8842998926765759-OFF-using-model-sem-lex.npy
* 0.80976224 5666794693555826-COOL_3-using-model-sem-lex.npy
* 0.80927503 5941139621427864-ENVELOPE-using-model-sem-lex.npy
3
Query File: 1989909351732737-FU

array([[    0, 76462,  2889,  4184],
       [    1,  2566,  5324, 26912],
       [    2, 35168, 19509, 72424],
       [    3, 61666, 51239, 26410],
       [    4, 47227, 54058, 19592]])

In [143]:
sail_df = recombined_df[recombined_df['file_name'].str.contains("SAIL_2", na=False)]
sail_df

,video_id,file_path,file_name,embedding
6,029833014387566248,/opt/home/cleong/data/ASL_Citizen/embeddings/0...,029833014387566248-SAIL_2-using-model-sem-lex.npy,"[-0.20589952, -0.4929245, -0.405769, 0.4774660..."
52,25193493599062466,/opt/home/cleong/data/ASL_Citizen/embeddings/2...,25193493599062466-SAIL_2-using-model-sem-lex.npy,"[0.020708017, -0.64416075, -0.571725, 0.414683..."
1104,998664525579968,/opt/home/cleong/data/ASL_Citizen/embeddings/9...,998664525579968-SAIL_2-using-model-sem-lex.npy,"[-0.028152522, -0.17866078, -0.40053686, 0.612..."
3232,3561548719390817,/opt/home/cleong/data/ASL_Citizen/embeddings/3...,3561548719390817-SAIL_2-using-model-sem-lex.npy,"[-0.07449641, 0.14802885, -0.15747677, 0.63058..."
5293,271562403427132,/opt/home/cleong/data/ASL_Citizen/embeddings/2...,271562403427132-SAIL_2-using-model-sem-lex.npy,"[0.11556907, -0.47648573, -0.069870405, 0.0706..."
9224,6963257268664678,/opt/home/cleong/data/ASL_Citizen/embeddings/6...,6963257268664678-SAIL_2-using-model-sem-lex.npy,"[-0.15684293, -0.17237327, -0.2684914, 0.72625..."
9864,3037369286672158,/opt/home/cleong/data/ASL_Citizen/embeddings/3...,3037369286672158-SAIL_2-using-model-sem-lex.npy,"[0.14184958, -0.1991123, -0.38218364, 0.547064..."
13270,7450908595041557,/opt/home/cleong/data/ASL_Citizen/embeddings/7...,7450908595041557-SAIL_2-using-model-sem-lex.npy,"[-0.3905956, -0.17760131, -0.19289908, 0.68476..."
13358,577224478912544,/opt/home/cleong/data/ASL_Citizen/embeddings/5...,577224478912544-SAIL_2-using-model-sem-lex.npy,"[-0.13551623, 0.08971065, -0.31820688, 0.47439..."
14201,25978172118606313,/opt/home/cleong/data/ASL_Citizen/embeddings/2...,25978172118606313-SAIL_2-using-model-sem-lex.npy,"[0.03173498, 0.15703946, 0.034203187, 0.459587..."


In [144]:
search_and_output_indices(sail_df)


0
Query File: 029833014387566248-SAIL_2-using-model-sem-lex.npy
* 1.0 029833014387566248-SAIL_2-using-model-sem-lex.npy
* 0.9211222 7991265889046182-SAILBOAT-using-model-sem-lex.npy
* 0.91164017 19884607898478812-SAIL_2-using-model-sem-lex.npy
* 0.89361244 5526703075746617-SAIL_2-using-model-sem-lex.npy
1
Query File: 25193493599062466-SAIL_2-using-model-sem-lex.npy
* 1.0 25193493599062466-SAIL_2-using-model-sem-lex.npy
* 0.8849796 732802596992111-MORNING-using-model-sem-lex.npy
* 0.8836915 5791417290435064-ERUPT_1-using-model-sem-lex.npy
* 0.8728297 2115074159591035-SAILBOAT-using-model-sem-lex.npy
2
Query File: 998664525579968-SAIL_2-using-model-sem-lex.npy
* 1.0000001 998664525579968-SAIL_2-using-model-sem-lex.npy
* 0.91859174 43820531816223984-SAILBOAT-using-model-sem-lex.npy
* 0.9121847 10481713339451981-CAMCORDER-using-model-sem-lex.npy
* 0.9060828 501034080069992-SAILBOAT-using-model-sem-lex.npy
3
Query File: 3561548719390817-SAIL_2-using-model-sem-lex.npy
* 1.0000001 3561548719

array([[    6, 73882, 26826, 38394],
       [   52, 75575, 19559, 43675],
       [ 1104, 21763, 53395, 52713],
       [ 3232,  7065, 70953, 13358],
       [ 5293,  9693, 59365, 76958],
       [ 9224, 64801, 70953, 47402],
       [ 9864, 61945, 28003, 19493],
       [13270, 54929, 74346, 64767],
       [13358, 47402, 54929,  7065],
       [14201, 57969, 27621, 31570],
       [14608, 52124, 67136, 31766],
       [15144, 70053, 51773, 43111],
       [16053, 60564, 73760, 73567],
       [20835, 22632,  7759,  5293],
       [26826, 73882,  1668, 39462],
       [32199, 59741, 14201, 74413],
       [37108, 33162, 61525, 29194],
       [38394,     6, 21763, 39590],
       [39462, 58683, 38106, 73882],
       [41717, 29181, 60564, 73882],
       [43514, 15150, 21726, 19511],
       [47402, 13358,  7065, 70953],
       [48017,  5293, 50653,  9693],
       [54313, 11970, 52870, 72253],
       [54929, 74346, 13270,  7065],
       [58683, 39462, 38106, 73882],
       [61622, 48178, 41061, 57033],
 

PCA and Clustering
https://github.com/facebookresearch/faiss/wiki/Faiss-building-blocks:-clustering,-PCA,-quantization

In [154]:
def do_pca(mt, d=768, d_out=10):
    # random training data 
    # mt = np.random.rand(1000, d).astype('float32')
    mat = faiss.PCAMatrix (d, d_out)
    mat.train(mt)
    assert mat.is_trained
    tr = mat.apply(mt)
    return tr
    # print this to show that the magnitude of tr's columns is decreasing
    # print (tr ** 2).sum(0)
pca = do_pca(get_embeddings_array_from_df(recombined_df))
# print(tr.sum())

In [156]:
pca.shape

(83112, 10)